In [1]:
import os
import mlflow
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    auc
)


/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
X_val = pd.read_csv("../data/processed/raw_features/X_val.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()
y_val = pd.read_csv("../data/processed/target/y_val.csv").squeeze()

X = X_train.copy()
X['no_alance'] = X['Balance']==0
X['Balance_equal_width_bins'] = pd.cut(X['Balance'], bins=7)
X['Balance_equal_frequency_bins'] = pd.qcut(X['Balance'], q=7)
X['age_equal_width_bins'] = pd.cut(X['Age'], bins=6)
X['age_equal_frequency_bins'] = pd.qcut(X['Age'], q=6)
X['Salary_equal_width_bins'] = pd.cut(X['EstimatedSalary'], bins=6)
X['Salary_equal_frequency_bins'] = pd.qcut(X['EstimatedSalary'], q=6)
X['NumOfProducts_2'] = X['NumOfProducts']==2
X['NumOfProducts_1'] = X['NumOfProducts']==1
X['NumOfProducts_3_4'] = X['NumOfProducts'].isin([3, 4])
X = X.drop(columns=['NumOfProducts'])

In [3]:
class FeatureEngineer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # -----------------------------
        # binary
        # -----------------------------
        X["no_balance"] = (
            X["Balance"] == 0
            )

        X["NumOfProducts_1"] = (
            X["NumOfProducts"] == 1
            )

        X["NumOfProducts_2"] = (
            X["NumOfProducts"] == 2
            )
        
        # -----------------------------
        # interactions
        # -----------------------------
        X["Balance_x_Tenure"] = (
            X["Balance"] * X["Tenure"]
            )

        X["Age_x_IsActive"] = (
            X["Age"] * X["IsActiveMember"]
            )

        # -----------------------------
        # ratios
        # -----------------------------
        X["Balance_to_Salary"] = (
            X["Balance"] /
            (X["EstimatedSalary"] + 1)
            )

        X["Balance_per_Product"] = (
            X["Balance"] /
            (X["NumOfProducts"] + 1)
            )

        # Customer value & engagement features

        X["Salary_per_Product"] = (
            X["EstimatedSalary"] / 
            (X["NumOfProducts"] + 1)
            )

        X["CreditScore_per_Age"] = (
            X["CreditScore"] / 
            (X["Age"] + 1)
            )

        X["Tenure_per_Age"] = (
            X["Tenure"] / 
            (X["Age"] + 1)
            )

        # Behavioral risk features

        X["Inactive_x_Balance"] = (
            (1 - X["IsActiveMember"]) * X["Balance"]
            )

        X["Inactive_x_Age"] = (
            (1 - X["IsActiveMember"]) * X["Age"]
            )

        X["Products_x_Active"] = (
            X["NumOfProducts"] * X["IsActiveMember"]
            )

        # Financial intensity features

        X["Balance_plus_Salary"] = (
            X["Balance"] + X["EstimatedSalary"]
            )

        X["WealthScore"] = (
            0.6 * X["Balance"] +
            0.4 * X["EstimatedSalary"]
            )

        X["CreditScore_x_Age"] = (
            X["CreditScore"] * X["Age"]
            )

        X["LogBalance"] = np.log1p(X["Balance"])

        X["LogAge"] = np.log1p(X["Age"])

        # Nonlinear continuous transforms

        X["Age2"] = X["Age"] ** 2

        X["Balance2"] = X["Balance"] ** 2

        X["Tenure2"] = X["Tenure"] ** 2

        # Product behavior features

        X["Products_per_Tenure"] = (
            X["NumOfProducts"] / (X["Tenure"] + 1)
        )

        X["Balance_per_Tenure"] = (
            X["Balance"] / (X["Tenure"] + 1)
        )

        return X

In [4]:
# =========================================================
# Columns
# =========================================================

nomod_columns = [
    'HasCrCard',
    'IsActiveMember',
]

dummyfy_columns = [
    'Card Type',
    'NumOfProducts',
    'Geography',
    'Gender'
]

norm_std_columns = [
    'Balance',
    'Point Earned',
    'CreditScore',
    'Age',
    'Tenure',
    'Satisfaction Score',
    'EstimatedSalary'
]

In [5]:
new_feature_engineered_columns = [
    # binary
    "no_balance",  # 0.721986
    "NumOfProducts_1", # 0.723806
    "NumOfProducts_2", # 0.711833

    # interactions
    "Balance_x_Tenure",
    "Age_x_IsActive",

    # ratios
    "Balance_to_Salary",
    "Balance_per_Product",
    "Salary_per_Product",
    "CreditScore_per_Age",
    "Tenure_per_Age",

    # behavioral risk
    "Inactive_x_Balance",
    "Inactive_x_Age",
    "Products_x_Active",

    # financial intensity
    "Balance_plus_Salary",
    "WealthScore",
    "CreditScore_x_Age",

    # log transforms
    "LogBalance",
    "LogAge",

    # nonlinear transforms
    "Age2",
    "Balance2",
    "Tenure2",

    # product behavior
    "Products_per_Tenure",
    "Balance_per_Tenure"
]

In [6]:
# =========================================================
# Config
# =========================================================

RANDOM_STATE = 42
EXPERIMENT_NAME = "customer-churn-simple-feature-engineering"

# =========================================================
# Paths
# =========================================================

ROOT_DIR = os.path.abspath("../")

DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")

ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

# =========================================================
# MLflow
# =========================================================

mlflow.set_tracking_uri(
    f"sqlite:///{DB_PATH}"
)

# Create experiment only once
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:

    mlflow.create_experiment(
        name=EXPERIMENT_NAME,
        artifact_location=f"file://{ARTIFACTS_DIR}"
    )

mlflow.set_experiment(EXPERIMENT_NAME)



<Experiment: artifact_location='file:///Users/hector.vargas/repos/ml_hands_on_project/mlartifacts', creation_time=1779234383129, experiment_id='3', last_update_time=1779234383129, lifecycle_stage='active', name='customer-churn-simple-feature-engineering', tags={}, trace_location=None, workspace='default'>

In [7]:
# =========================================================
# Preprocessor
# =========================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                drop='first'
            ),
            dummyfy_columns
        ),
        (
            'num',
            StandardScaler(),
            norm_std_columns
        ),
        (
            'pass',
            'passthrough',
            nomod_columns
        )
    ],
    remainder='drop'
)

# =========================================================
# Models
# =========================================================

models = {
    "random_forest": RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

In [8]:
# =========================================================
# Helper: get names
# =========================================================

def get_feature_names(preprocessor):
    return preprocessor.get_feature_names_out()

# =========================================================
# Helper: Log Curves
# =========================================================

def log_classification_curves(y_test, y_proba):

    # --------------------------------
    # ROC Curve
    # --------------------------------

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend(loc="lower right")

    mlflow.log_figure(plt.gcf(), "roc_curve.png")
    plt.close()

    # --------------------------------
    # Precision Recall Curve
    # --------------------------------

    precision, recall, _ = precision_recall_curve(
        y_test,
        y_proba
    )

    pr_auc = average_precision_score(y_test, y_proba)

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"PR AUC = {pr_auc:.4f}")

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend(loc="lower left")

    mlflow.log_figure(plt.gcf(), "pr_curve.png")
    plt.close()

In [9]:
# =========================================================
# Train Log
# =========================================================
def train_log(models, preprocessor, X_train, y_train, X_test, y_test):
    for model_name, model in models.items():
        with mlflow.start_run(run_name=model_name):

            # --------------------------------
            # Build Pipeline
            # --------------------------------

            pipeline = Pipeline([
                #("feature_engineering", FeatureEngineer()),
                ("preprocessing", preprocessor),
                ("model", model)
            ])

            # --------------------------------
            # Train
            # --------------------------------

            pipeline.fit(X_train, y_train)

            # --------------------------------
            # Predictions
            # --------------------------------

            y_pred = pipeline.predict(X_test)

            if hasattr(pipeline, "predict_proba"):

                y_proba = pipeline.predict_proba(X_test)[:, 1]

                roc_auc = roc_auc_score(y_test, y_proba)

                pr_auc = average_precision_score(
                    y_test,
                    y_proba
                )

                # Log plots
                log_classification_curves(
                    y_test,
                    y_proba
                )

            else:
                roc_auc = None
                pr_auc = None

            # --------------------------------
            # Metrics
            # --------------------------------

            metrics = {
                "accuracy": accuracy_score(y_test, y_pred),
                "precision": precision_score(y_test, y_pred),
                "recall": recall_score(y_test, y_pred),
                "f1_score": f1_score(y_test, y_pred)
            }

            if roc_auc is not None:
                metrics["roc_auc"] = roc_auc

            if pr_auc is not None:
                metrics["pr_auc"] = pr_auc

            mlflow.log_metrics(metrics)

            # --------------------------------
            # Params
            # --------------------------------

            mlflow.log_params(model.get_params())

            # --------------------------------
            # Feature Schema
            # --------------------------------

            fitted_preprocessor = pipeline.named_steps['preprocessing']

            input_features = (
                dummyfy_columns +
                norm_std_columns +
                nomod_columns
            )

            output_features = get_feature_names(
                fitted_preprocessor
            )

            mlflow.log_dict(
                {
                    "input_features": input_features,
                    "output_features": output_features.tolist()
                },
                "feature_schema.json"
            )

            # --------------------------------
            # Reports
            # --------------------------------

            mlflow.log_dict(
                classification_report(
                    y_test,
                    y_pred,
                    output_dict=True
                ),
                "classification_report.json"
            )

            mlflow.log_dict(
                confusion_matrix(
                    y_test,
                    y_pred
                ).tolist(),
                "confusion_matrix.json"
            )

            # --------------------------------
            # Log Model
            # --------------------------------

            mlflow.sklearn.log_model(
                sk_model=pipeline,
                name="model",
                serialization_format="cloudpickle"
            )

            print(f"Finished: {model_name}")
            print(metrics["pr_auc"])

In [10]:
train_log(models, preprocessor, X_train, y_train, X_test, y_test)

2026/05/21 18:21:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: random_forest
0.7333870674652824


In [11]:
def experiment_metrics(experiment_name):
    # --------------------------------
    # Get Experiment
    # --------------------------------

    experiment = mlflow.get_experiment_by_name(experiment_name)
    experiment_id = experiment.experiment_id

    # --------------------------------
    # Search Runs
    # --------------------------------

    runs_df = mlflow.search_runs(
        experiment_ids=[experiment_id]
    )

    # Display summary table
    summary_df = runs_df[[
            "run_id",
            "tags.mlflow.runName",
            "metrics.accuracy",
            "metrics.precision",
            "metrics.recall",
            "metrics.f1_score",
            "metrics.roc_auc",
            "metrics.pr_auc",
            "start_time"
        ]]

    # --------------------------------
    # Sort by Best Model
    # --------------------------------

    summary_df = summary_df.sort_values(
        by="metrics.pr_auc",
        ascending=False
    )


    # --------------------------------
    # Display
    # --------------------------------

    display(summary_df)

In [12]:
experiment_metrics(EXPERIMENT_NAME)

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,metrics.pr_auc,start_time
0,4b5106d1622f4e68a2c164aa87eff1de,random_forest,0.8755,0.795539,0.524510,0.632201,0.875317,0.733387,2026-05-22 00:21:19.296000+00:00
1,e5e851084e134521b2ea535447feb7b1,random_forest,0.8755,0.795539,0.524510,0.632201,0.875317,0.733387,2026-05-21 23:56:54.726000+00:00
5,571d5d0a719540e18eb506e9775c9684,random_forest,0.8755,0.795539,0.524510,0.632201,0.875317,0.733387,2026-05-21 23:43:53.173000+00:00
3,1a05df46f9e5449ebca78510d2ff6f71,random_forest,0.8755,0.791209,0.529412,0.634361,0.872341,0.732173,2026-05-21 23:56:06.072000+00:00
8,dc4712cc623a48dd8f314883be8e57bd,random_forest,0.8755,0.795539,0.524510,0.632201,0.873903,0.730011,2026-05-21 23:41:35.659000+00:00
12,e78617ca78e24dac97114b190bbf838f,random_forest,0.8770,0.809160,0.519608,0.632836,0.869601,0.729070,2026-05-21 23:39:43.807000+00:00
14,4218dceac02446419cd25f0d0b244c13,random_forest,0.8755,0.802281,0.517157,0.628912,0.870775,0.728192,2026-05-21 23:38:07.381000+00:00
10,ac01ffc738c644f59d0ae3b1c20719b7,random_forest,0.8740,0.788889,0.522059,0.628319,0.867435,0.728187,2026-05-21 23:40:49.973000+00:00
6,3011b6dd2286443f85e723870a2a5c11,random_forest,0.8770,0.797794,0.531863,0.638235,0.873361,0.727793,2026-05-21 23:43:29.509000+00:00
11,01e90724747048f8aee03a9d89f911ee,random_forest,0.8705,0.787645,0.500000,0.611694,0.872255,0.726353,2026-05-21 23:40:20.015000+00:00


In [13]:
def remove_mlflow_experiments(number_of_experiments_to_remove):
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    experiment_id = experiment.experiment_id

    client = mlflow.tracking.MlflowClient()

    # Get runs ordered by newest first
    runs = mlflow.search_runs(
        experiment_ids=[experiment_id],
        order_by=["start_time DESC"]
    )

    # Select last 5 runs
    last_5_runs = runs.head(number_of_experiments_to_remove)

    # Delete them
    for run_id in last_5_runs["run_id"]:
        client.delete_run(run_id)
        print(f"Deleted run: {run_id}")

In [14]:
#remove_mlflow_experiments(6)

In [15]:
from copy import deepcopy
from sklearn.base import clone


def evaluate_engineered_features(
    engineered_features,
    base_nomod_columns,
    dummyfy_columns,
    norm_std_columns,
    model,
    X_train,
    y_train,
    X_test,
    y_test
):
    results = []

    # =====================================================
    # Baseline
    # =====================================================

    baseline_preprocessor = ColumnTransformer(
        transformers=[
            (
                'cat',
                OneHotEncoder(
                    handle_unknown='ignore',
                    drop='first'
                ),
                dummyfy_columns
            ),
            (
                'num',
                StandardScaler(),
                norm_std_columns
            ),
            (
                'pass',
                'passthrough',
                base_nomod_columns
            )
        ],
        remainder='drop'
    )

    baseline_pipeline = Pipeline([
        ("feature_engineering", FeatureEngineer()),
        ("preprocessing", baseline_preprocessor),
        ("model", clone(model))
    ])

    baseline_pipeline.fit(X_train, y_train)

    baseline_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "feature_added": "BASELINE",
        "pr_auc": average_precision_score(
            y_test,
            baseline_proba
        )
    })

    # =====================================================
    # One feature at a time
    # =====================================================

    for feature in engineered_features:

        current_nomod = deepcopy(norm_std_columns)
        current_nomod.append(feature)

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    'cat',
                    OneHotEncoder(
                        handle_unknown='ignore',
                        drop='first'
                    ),
                    dummyfy_columns
                ),
                (
                    'num',
                    StandardScaler(),
                    current_nomod
                ),
                (
                    'pass',
                    'passthrough',
                    base_nomod_columns
                )
            ],
            remainder='drop'
        )

        pipeline = Pipeline([
            ("feature_engineering", FeatureEngineer()),
            ("preprocessing", preprocessor),
            ("model", clone(model))
        ])

        pipeline.fit(X_train, y_train)

        y_proba = pipeline.predict_proba(X_test)[:, 1]

        results.append({
            "feature_added": feature,
            "pr_auc": average_precision_score(
                y_test,
                y_proba
            )
        })

    # =====================================================
    # Rank
    # =====================================================

    results_df = (
        pd.DataFrame(results)
        .sort_values(by="pr_auc", ascending=False)
        .reset_index(drop=True)
    )

    return results_df

In [16]:
results_df = evaluate_engineered_features(
    engineered_features=new_feature_engineered_columns,
    base_nomod_columns=nomod_columns,
    dummyfy_columns=dummyfy_columns,
    norm_std_columns=norm_std_columns,
    model=models["random_forest"],
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test
)

display(results_df)

,feature_added,pr_auc
0,BASELINE,0.733387
1,Age_x_IsActive,0.732918
2,NumOfProducts_1,0.731978
3,Inactive_x_Age,0.731502
4,Balance_per_Product,0.730197
5,Tenure2,0.729435
6,Salary_per_Product,0.729184
7,Inactive_x_Balance,0.727405
8,Balance_per_Tenure,0.727174
9,Products_x_Active,0.727110
